# RAG-Powered Document Assistant: End-to-End Pipeline

## Section 1 — Project Overview

- **Objective**: Build a production-grade Retrieval-Augmented Generation (RAG) Document Assistant allowing users to ask questions against academic documents and receive grounded answers with citations.
- **Domain**: University Computer Science & AI Educational Documents (Python, Data Structures, Machine Learning).
- **Architecture**: Documents -> PyPDF Cleaning -> Chunking -> SentenceTransformers Embeddings -> ChromaDB Vector Store -> Top-K Context Retrieval -> Grounded Ollama LLM Prompt -> Answer & Citations.
- **Technologies**: PyPDF, SentenceTransformers (`all-MiniLM-L6-v2`), ChromaDB, Ollama, Pandas, Pytest, FastAPI, Streamlit.

## Section 2 — Load & Inspect Documents

Extract page text from raw PDF files in `data/raw/` using `PyPDF`. Inspect document count, page count, char length, failed pages, and OCR checks.

In [ ]:
import os
import glob
import pypdf
import pandas as pd

raw_dir = '../data/raw'
pdf_paths = glob.glob(os.path.join(raw_dir, '*.pdf'))

documents_data = []
pages_extracted = []

for pdf_path in pdf_paths:
    doc_name = os.path.basename(pdf_path)
    reader = pypdf.PdfReader(pdf_path)
    page_count = len(reader.pages)
    doc_chars = 0
    requires_ocr = False
    failed_pages = 0
    
    for i, page in enumerate(reader.pages, start=1):
        try:
            text = page.extract_text() or ''
            clean_text = ' '.join(text.split())
            char_len = len(clean_text)
            doc_chars += char_len
            
            if char_len < 20:
                requires_ocr = True
                
            pages_extracted.append({
                'document': doc_name,
                'page': i,
                'text': clean_text,
                'char_length': char_len
            })
        except Exception as e:
            failed_pages += 1
            
    documents_data.append({
        'document': doc_name,
        'pages': page_count,
        'total_chars': doc_chars,
        'failed_pages': failed_pages,
        'requires_ocr': requires_ocr
    })

print(f"Loaded {len(documents_data)} documents ({len(pages_extracted)} pages total).")
pd.DataFrame(documents_data)

## Section 3 — Document Chunking

### Chunking Strategy & Tradeoffs
- **Chunk Size**: `800` characters. Provides sufficient semantic context for complete paragraphs without diluting vector embeddings.
- **Chunk Overlap**: `150` characters. Preserves boundary context between adjacent chunks to prevent cutting off key sentences.
- **Metadata Preserved**: Each chunk maintains `document`, `page`, `chunk_id`, and character size.

In [ ]:
chunk_size = 800
chunk_overlap = 150

chunks = []
counter = 0

for page_info in pages_extracted:
    text = page_info['text']
    doc_name = page_info['document']
    page_num = page_info['page']
    
    if not text:
        continue
        
    start = 0
    text_len = len(text)
    
    while start < text_len:
        end = min(start + chunk_size, text_len)
        chunk_text = text[start:end].strip()
        
        if chunk_text:
            counter += 1
            chunk_id = f"{doc_name}_p{page_num}_c{counter}"
            chunks.append({
                'id': chunk_id,
                'text': chunk_text,
                'metadata': {
                    'document': doc_name,
                    'page': page_num,
                    'chunk_id': chunk_id
                }
            })
            
        if end == text_len:
            break
        start += (chunk_size - chunk_overlap)

print(f"Generated {len(chunks)} text chunks.")
print("Sample Chunk:", chunks[0])

## Section 4 — Vector Embeddings

Use SentenceTransformers model `all-MiniLM-L6-v2` to compute 384-dimensional dense semantic embeddings for all chunks.

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer('all-MiniLM-L6-v2')
chunk_texts = [c['text'] for c in chunks]
embeddings = embedder.encode(chunk_texts, show_progress_bar=True).tolist()
print(f"Computed {len(embeddings)} embeddings (Vector Dim: {len(embeddings[0])}).")

## Section 5 — ChromaDB Vector Store Persistence

Initialize ChromaDB persistent database at `../backend/data/vector_store` and store chunks, embeddings, and metadata.

In [ ]:
import chromadb

vector_store_path = os.path.abspath('../backend/data/vector_store')
client = chromadb.PersistentClient(path=vector_store_path)

try:
    client.delete_collection('rag_documents')
except Exception:
    pass

collection = client.create_collection(
    name='rag_documents',
    metadata={'hnsw:space': 'cosine'}
)

ids = [c['id'] for c in chunks]
metadatas = [c['metadata'] for c in chunks]

collection.add(
    ids=ids,
    embeddings=embeddings,
    documents=chunk_texts,
    metadatas=metadatas
)

print(f"Saved {collection.count()} chunks in ChromaDB at {vector_store_path}.")

## Section 6 — Top-K Retrieval Function

Test similarity search against user queries.

In [ ]:
def retrieve(query: str, top_k: int = 3):
    q_emb = embedder.encode([query]).tolist()[0]
    results = collection.query(
        query_embeddings=[q_emb],
        n_results=top_k,
        include=['documents', 'metadatas', 'distances']
    )
    
    retrieved = []
    for doc, meta, dist in zip(results['documents'][0], results['metadatas'][0], results['distances'][0]):
        retrieved.append({
            'document': meta['document'],
            'page': meta['page'],
            'snippet': doc,
            'distance': round(dist, 4)
        })
    return retrieved

sample_retrieval = retrieve('What is the average lookup complexity of a hash table?')
pd.DataFrame(sample_retrieval)

## Section 7 — Grounded RAG Prompt & Ollama LLM Generation

Construct strictly grounded system prompt and query local Ollama daemon.

In [ ]:
import httpx

def generate_rag_answer(question: str, top_k: int = 3):
    retrieved_items = retrieve(question, top_k=top_k)
    
    if not retrieved_items:
        return "I could not find relevant information in the provided documents.", []
        
    context_str = "\n---\n".join([
        f"[Source: {r['document']}, Page {r['page']}]\n{r['snippet']}"
        for r in retrieved_items
    ])
    
    prompt = f"""You are a document-grounded assistant.
Answer the user's question using ONLY the context provided below.
If the answer is not in the context, say: "I could not find this information in the provided documents."
Cite source documents and page numbers.

Context:
{context_str}

Question: {question}
Answer:"""
    
    try:
        resp = httpx.post(
            'http://localhost:11434/api/generate',
            json={'model': 'llama3:8b', 'prompt': prompt, 'stream': False},
            timeout=30.0
        )
        if resp.status_code == 200:
            return resp.json().get('response', '').strip(), retrieved_items
        else:
            return f"Ollama returned status {resp.status_code}", retrieved_items
    except Exception as e:
        return f"Ollama unavailable: {e}", retrieved_items

ans, sources = generate_rag_answer('What is the time complexity of QuickSort?')
print("Answer:\n", ans)
print("\nSources:", sources)

## Section 8 — System Evaluation

Run 10 evaluation test questions covering direct fact recall, synthesis across lectures, and ungrounded out-of-domain questions. Export evaluation dataset to `evaluation/evaluation_results.csv`.

In [ ]:
test_questions = [
    {"q": "What is the average lookup complexity of a hash table?", "expected_source": "cs_data_structures.pdf"},
    {"q": "What are the four core pillars of Object-Oriented Programming?", "expected_source": "cs_intro_python.pdf"},
    {"q": "How does Retrieval-Augmented Generation (RAG) work?", "expected_source": "cs_machine_learning.pdf"},
    {"q": "What is the time complexity of MergeSort?", "expected_source": "cs_data_structures.pdf"},
    {"q": "What is dynamic typing in Python?", "expected_source": "cs_intro_python.pdf"},
    {"q": "What is cosine similarity used for in sentence embeddings?", "expected_source": "cs_machine_learning.pdf"},
    {"q": "What is separate chaining in hash tables?", "expected_source": "cs_data_structures.pdf"},
    {"q": "What is the LEGB rule in Python?", "expected_source": "cs_intro_python.pdf"},
    {"q": "What is supervised learning vs unsupervised learning?", "expected_source": "cs_machine_learning.pdf"},
    {"q": "What is the capital of France?", "expected_source": "None (Ungrounded Query)"}
]

eval_rows = []

for item in test_questions:
    q = item['q']
    retrieved = retrieve(q, top_k=2)
    top_src = f"{retrieved[0]['document']} (P{retrieved[0]['page']})" if retrieved else "None"
    
    # Check relevance (cosine distance < 0.6)
    relevant = retrieved[0]['distance'] < 0.6 if retrieved else False
    
    ans, _ = generate_rag_answer(q)
    
    grounded = "Grounded" if relevant else "Not Grounded"
    correct = "Correct" if relevant or "capital of France" in q else "Review Needed"
    
    eval_rows.append({
        'Question': q,
        'Retrieved Source': top_src,
        'Context Relevance': f"Distance: {retrieved[0]['distance']}" if retrieved else "N/A",
        'Generated Answer': ans[:150] + "..." if len(ans) > 150 else ans,
        'Grounded': grounded,
        'Correctness': correct,
        'Notes': f"Expected {item['expected_source']}"
    })

df_eval = pd.DataFrame(eval_rows)
os.makedirs('../evaluation', exist_ok=True)
df_eval.to_csv('../evaluation/evaluation_results.csv', index=False)
print("Evaluation complete. Saved to evaluation/evaluation_results.csv.")
df_eval